# Normal-text probe -- does K5 care that "Normal" says *surveillance*?

Rule: `core/docs/final/KIM_RUN.md` §15, written before any number. **Inference only on K5 s2024 + s2025, nothing is trained, K5 stays the main model.** Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, preflight (K5 ckpts, their recorded reads, bakes) | no | 2-3 min |
| 4 | tests on the probe -- **raises if red** | no | 1 min |
| 5 | **step 1**: `adp_alpha` -> H_bin's pre-fusion weight `w`, both seeds | no | 1 min |
| 6 | **step 2**: K5 x {DoTA-CAP-dev, DADA} x {original (gate), dashcam} x 2 seeds = 8 reads | **yes** | 30-60 min |
| 7 | report (gate + verdict) -> **paste `REPORTS/normal_text_probe/normal_text_probe.md` to Claude** | no | seconds |

Every read skips what is already on Drive. Thesis and Thesis-V2 are only read; everything new goes to `Thesis-Final/outputs/kim/REPORTS/normal_text_probe/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: A2 bakes, Nexar_window defs
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5 + its reads; the probe's outputs go here
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert (REPO / 'core/tools/normal_text_probe.py').is_file(), 'this upload has no normal_text_probe -- re-upload kat-vad/'
assert '## 15. Normal-text probe' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §15 -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5       = constants.KIM_MAIN_ARM                  # 'K5'
SEEDS    = (constants.KIM_SEED, constants.KIM_REPLICATE_SEED)   # 2024, 2025
ENC      = constants.V5_ENCODER                    # V2-S
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
V2_CACHE = Path(DRIVE_V2) / 'cache'
DATA     = Path(DRIVE_V2) / 'data' / f'{constants.NEXAR_DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'  # defs.json K5 trained with
A2_DADA  = V2_CACHE / 'v2' / 'A2_S' / f'{T2NAME}_test_sources_nexar'
A2_DOTA  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
ROOT     = constants.OUTPUT_ROOT / constants.KIM_OUTPUT_DIRNAME    # Thesis-Final/outputs/kim
OUT      = ROOT / 'REPORTS' / constants.NORMAL_PROBE_REPORT_DIRNAME
REPORT   = OUT / 'normal_text_probe.md'
ALPHA    = OUT / 'alpha.json'
LOCAL    = Path('/content/normal_probe')
LOCAL.mkdir(parents=True, exist_ok=True)


def eval_root(seed):
    return ROOT / ('eval' if seed == constants.KIM_SEED else f'eval_s{seed}')   # as kim_seed.ipynb


CKPT     = {s: ROOT / K5 / f's{s}' / 'checkpoint_last.pt' for s in SEEDS}
RECORDED = {(st, s): eval_root(s) / st / K5 for st in ('dota', 'dada') for s in SEEDS}
READOUT  = {'dota': 'protocol_b_readout.json', 'dada': 'dada_readout.json'}

for name, path in (('Nexar defs', DATA / constants.DEFS_FILENAME), ('DoTA-CAP A2 bake', A2_DOTA), ('DADA A2 bake', A2_DADA),
                   ('T2 meta', T2 / constants.META_FILENAME), ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT),
                   ('DoTA s1', DOTA_S1), ('DoTA metadata', DOTA / 'metadata_val.json'), ('DoTA split', DOTA / 'val_split.txt'),
                   ('DoTA labels', DOTA / 'labels_s8'),
                   *((f'K5 s{s} ckpt', CKPT[s]) for s in SEEDS),
                   *((f'K5 {st} s{s} read', d / READOUT[st]) for (st, s), d in RECORDED.items())):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:22s} {path}')
defs = json.loads((DATA / constants.DEFS_FILENAME).read_text())
assert defs[0] == constants.NORMAL_PROBE_CLASS, f'defs.json does not put Normal first: {defs}'
for (st, s), d in RECORDED.items():                # the recorded read is this checkpoint's own
    r = json.loads((d / READOUT[st]).read_text())
    assert list(r['checkpoints']) == [f's{s}'] and Path(r['checkpoints'][f's{s}']) == CKPT[s], f'{st} s{s} read is not K5 s{s}: {r["checkpoints"]}'
    print(f'  recorded {st} s{s}: macro {r["macro"]["mean"]:.4f}, step {r["global_steps"][f"s{s}"]}')
print(f'class list {defs} | report -> {OUT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 6 needs one'

In [ ]:
import torch

from core.data.v2_splits import load_split           # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
    drive.mount('/content/drive')


DADA_IDS = test_sources(T2)
CAP_IDS = sorted(load_split(CAP_DEV))
assert len(DADA_IDS) == 370, len(DADA_IDS)
for name, d, ids in (('DADA', A2_DADA, DADA_IDS), ('DoTA-CAP', A2_DOTA, CAP_IDS)):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} {name} videos have no A2 bake: {miss[:5]}'
    m = json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
    assert (m['crn'], m['motion']) == (constants.V2_OFF, ENC), f'{name} bake is {m}'
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in DATA.glob('*.json'):               # defs.json etc.: the definitions K5 trained with
    shutil.copy2(f, L_DEFS / f.name)
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
print(f'preflight OK: {len(DADA_IDS)} DADA, {len(CAP_IDS)} DoTA-CAP-dev videos')

## 4. Tests on the probe (CPU) -- **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_normal_text_probe.py'])

## 5. Step 1: H_bin's learned blend (CPU)

`w` = weight of the branch that never sees text; `1 - w` = the branch fed by co-attention with the Normal / CarAccident text. Init `w` = 0.924.

In [ ]:
if not ALPHA.exists():
    local = LOCAL / 'alpha.json'
    run([sys.executable, '-m', 'core.tools.normal_text_probe', 'alpha',
         *[x for s in SEEDS for x in ('--run', f's{s}', CKPT[s])], '--out', local])
    publish(local, ALPHA)
a = json.loads(ALPHA.read_text())
for name, r in a['runs'].items():
    print(f"{name}: step {r['global_step']} | alpha {r['alpha']:+.4f} | pre-fusion w {r['w_prefusion']:.4f} | text branch {r['w_text_branch']:.4f}")
print(f">>> text branch minor (w >= {a['minor_bar']}) at every seed: {a['text_branch_minor']}  -- printed, step 2 decides")

## 6. Step 2: 8 reads (GPU, inference only)

`original` is the gate (must reproduce the recorded K5 read), `dashcam` is the swap. Same verbalizer draws in both: only the Normal sentence differs.

In [ ]:
def score(set_name, seed, mode):
    out = OUT / 'scores' / set_name / f's{seed}' / mode
    if (out / 'clip_aucs.json').exists():
        print(f'{set_name} s{seed} {mode}: on Drive')
        return out
    assert torch.cuda.is_available(), 'torch sees no GPU'
    local = LOCAL / 'scores' / set_name / f's{seed}' / mode
    shutil.rmtree(local, ignore_errors=True)
    if set_name == 'dota':
        tool_args = ['--run', f's{seed}', CKPT[seed],
                     '--input-dir', stage(A2_DOTA, LOCAL / 'bake_dota', CAP_IDS), '--s1-dir', stage(DOTA_S1, LOCAL / 'dota_s1', CAP_IDS),
                     '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
                     '--data-dir', stage(DOTA / 'labels_s8', LOCAL / 'dota_labels'), '--split', CAP_DEV,
                     '--out-dir', local, '--device', 'auto']
    else:
        tool_args = ['--run', f's{seed}', CKPT[seed], '--t2-dir', L_T2, '--annotation', ANNOT, '--census', T2_CENSUS,
                     '--input-dir', stage(A2_DADA, LOCAL / 'bake_dada', DADA_IDS), '--data-dir', L_DEFS, '--out-dir', local]
    t0 = time.time()
    run([sys.executable, '-m', 'core.tools.normal_text_probe', 'score', '--tool', set_name, '--normal', mode, '--', *tool_args],
        log=LOCAL / f'{set_name}_s{seed}_{mode}.log')
    publish(local, out)
    print(f'{set_name} s{seed} {mode}: {(time.time() - t0) / 60:.1f} min')
    return out


SCORED = {}
for seed in SEEDS:
    for set_name in ('dota', 'dada'):
        for mode in ('original', 'dashcam'):    # original first: a gate failure shows before the swap is read
            SCORED[(set_name, seed, mode)] = score(set_name, seed, mode)
    flush_drive()

## 7. Report: gate + §15 verdict -> **paste `REPORTS/normal_text_probe/normal_text_probe.md` to Claude**

In [ ]:
local = LOCAL / 'report'
reads = [x for (st, s), d in RECORDED.items()
         for x in ('--read', st, f's{s}', d, SCORED[(st, s, 'original')], SCORED[(st, s, 'dashcam')])]
run([sys.executable, '-m', 'core.tools.normal_text_probe', 'report', '--alpha', ALPHA, *reads, '--out-dir', local])
publish(local, OUT)
flush_drive()
print(REPORT.read_text())
decision = json.loads((OUT / 'normal_text_probe.json').read_text())['decision']
print(f">>> verdict {decision['verdict']}: {decision['consequence']}")